## UAV Intrusion Detection Stacked Ensemble
This notebook uses One-Class SVM, Autoencoder, and Local Outlier Factor as base models, and combines their outputs with a meta-classifier ensemble.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import classification_report

### Load the Dataset

In [2]:
# Load the dataset
try:
    df = pd.read_excel("../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx")
except FileNotFoundError:
    print("Error: The file '../Datasets/excel/GPS_Data_Simplified_2D_Feature_Map.xlsx' was not found.")
    print("Please make sure the path is correct.")
    exit()
df.head()

,PRN,DO,PD,RX,TOW,CP,EC,LC,PC,PIP,PQP,TCD,CN0,Output
0,6,1160.671408,-238359.543282,491568.00,491568.000795,-24660.263293,163521.78,170008.81,179294.97,178420.33,-17688.17800,1158.4806,49.412529,0
1,6,1157.827808,-238363.921192,491568.02,491568.020795,-24683.468520,158277.05,160253.09,182106.11,182106.05,158.33333,1160.7948,49.452686,0
2,6,1161.135932,-238368.638615,491568.04,491568.040795,-24706.687357,184442.36,193125.89,208228.02,-205198.30,35391.59000,1157.6909,49.480145,0
3,6,1161.156617,-238373.391775,491568.06,491568.060795,-24729.920039,159812.45,169960.55,187550.72,187543.00,-1701.94400,1161.1343,49.433769,0
4,6,1160.690893,-238378.127066,491568.08,491568.080795,-24753.153999,183557.66,191912.58,208403.50,208330.05,-5532.89990,1161.1759,49.391670,0


### Preprocessing

In [3]:
# Prepare features and labels
X = df.drop('Output', axis=1)
y = df['Output']

# Use benign data to train unsupervised models
X_train_raw = X[y == 0]
X_test_raw = X
y_test = y

# Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# PCA for OC-SVM and LOF (85% variance)
pca_svm_lof = PCA(n_components=0.85, random_state=1082)
X_train_pca_svm_lof = pca_svm_lof.fit_transform(X_train_scaled)
X_test_pca_svm_lof = pca_svm_lof.transform(X_test_scaled)

# PCA for Autoencoder (95% variance)
pca_ae = PCA(n_components=0.95, random_state=1082)
X_train_pca_ae = pca_ae.fit_transform(X_train_scaled)
X_test_pca_ae = pca_ae.transform(X_test_scaled)

### Train Base Models

In [4]:
# One-Class SVM
ocsvm = OneClassSVM(nu=0.011, gamma=0.000211)
ocsvm.fit(X_train_pca_svm_lof)
ocsvm_score = ocsvm.decision_function(X_test_pca_svm_lof)  # anomaly score

# Local Outlier Factor
lof = LocalOutlierFactor(n_neighbors=1000, novelty=True)
lof.fit(X_train_pca_svm_lof)
lof_score = lof.decision_function(X_test_pca_svm_lof)  # anomaly score

# Autoencoder using MLP
ae = MLPRegressor(hidden_layer_sizes=(3,), activation='relu', max_iter=100, random_state=1082)
ae.fit(X_train_pca_ae, X_train_pca_ae)
recon = ae.predict(X_test_pca_ae)
ae_score = np.mean((X_test_pca_ae - recon) ** 2, axis=1)  # reconstruction error

### Meta-Classifier Ensemble

In [5]:
# Stack raw anomaly scores
stacked_features = np.vstack([ocsvm_score, lof_score, ae_score]).T

# Meta-classifier (Random Forest) for multiclass classification
meta_clf = RandomForestClassifier(random_state=1082)
meta_clf.fit(stacked_features, y_test)
stacked_pred = meta_clf.predict(stacked_features)

# Evaluation
print(classification_report(
    y_test, 
    stacked_pred, 
    labels=[0, 1, 2, 3],
    target_names=["benign", "attack1", "attack2", "attack3"],
    zero_division=0
))

              precision    recall  f1-score   support

      benign       0.99      1.00      0.99    397825
     attack1       0.98      0.95      0.97     36458
     attack2       1.00      0.99      1.00     44232
     attack3       0.99      0.95      0.97     32015

    accuracy                           0.99    510530
   macro avg       0.99      0.97      0.98    510530
weighted avg       0.99      0.99      0.99    510530

